In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt


from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Thay "HF_TOKEN" bằng đúng cái tên (Label) bạn đã đặt trong Add-ons
secret_label = "HF_TOKEN" 
secret_value = UserSecretsClient().get_secret(secret_label)

# 2. Dùng giá trị secret lấy được để đăng nhập tự động vào Hugging Face
login(token=secret_value, add_to_git_credential=False)

In [ ]:
# ============================================================
# CELL 0 — INSTALL
# ============================================================
!pip install -q "transformers>=4.46.0" accelerate qwen-vl-utilspython-Levenshtein


In [2]:
# ============================================================
# END-TO-END QA + ANLS + ERROR ANALYSIS
# Notebook mới — chạy sau khi có E0, E1, E2, E5
# ============================================================


# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, glob, json, zipfile, shutil, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from datetime import datetime
from tqdm.auto import tqdm
from Levenshtein import distance as levenshtein_distance  # pip install python-Levenshtein

from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor


# ============================================================
# CELL 2 — SETUP (robust, tìm artifact khắp input)
# ============================================================
WORK_ARTIFACT_DIR = "/kaggle/working/docvqa_colqwen2_artifacts"
RETRIEVAL_DIR     = f"{WORK_ARTIFACT_DIR}/retrieval_results"
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

DATASET_NAME = "vidore/docvqa_train"


def find_and_copy(target_path, filename_pattern, prefer_keyword=None):
    if os.path.exists(target_path):
        return True
    hits = glob.glob(f"/kaggle/input/**/{filename_pattern}", recursive=True)
    if not hits:
        return False
    if prefer_keyword:
        preferred = [h for h in hits if prefer_keyword in h.lower()]
        src = preferred[0] if preferred else hits[0]
    else:
        src = hits[0]
    shutil.copy(src, target_path)
    print(f"📄 Copied: {src}")
    return True


# Giải nén artifact nếu chưa có
MANIFEST_PATH = f"{WORK_ARTIFACT_DIR}/manifest.json"
if not os.path.exists(MANIFEST_PATH):
    all_zips = [
        p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if "docvqa_colqwen2_artifacts" in os.path.basename(p)
    ]
    assert all_zips, "❌ Không tìm thấy artifact zip"
    zip_path = max(all_zips, key=os.path.getsize)
    print(f"📦 Giải nén: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working")

# File cần thiết
REQUIRED = {
    "corpus_id_to_idx.json": f"{WORK_ARTIFACT_DIR}/corpus_id_to_idx.json",
    "e0_dense_topk_results.json": f"{RETRIEVAL_DIR}/e0_dense_topk_results.json",
    "e1_bm25_topk_results.json":  f"{RETRIEVAL_DIR}/e1_bm25_topk_results.json",
    "e2_rrf_topk_results.json":   f"{RETRIEVAL_DIR}/e2_rrf_topk_results.json",
    "e5_vlm_rerank_topk_results.json": f"{RETRIEVAL_DIR}/e5_vlm_rerank_topk_results.json",
    "e0_dense_metrics.json":      f"{RETRIEVAL_DIR}/e0_dense_metrics.json",
    "e1_bm25_metrics.json":       f"{RETRIEVAL_DIR}/e1_bm25_metrics.json",
    "e2_rrf_metrics.json":        f"{RETRIEVAL_DIR}/e2_rrf_metrics.json",
    "e5_vlm_rerank_metrics.json": f"{RETRIEVAL_DIR}/e5_vlm_rerank_metrics.json",
}
for fname, fpath in REQUIRED.items():
    ok = find_and_copy(fpath, fname)
    print(f"{'✅' if ok else '❌'} {fname}")

print("\n✅ Setup xong")


# ============================================================
# CELL 3 — LOAD TẤT CẢ KẾT QUẢ
# ============================================================
with open(f"{WORK_ARTIFACT_DIR}/corpus_id_to_idx.json", encoding="utf-8") as f:
    corpus_id_to_idx = json.load(f)

def load_results(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_metrics(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)["metrics"]

e0_results = load_results(f"{RETRIEVAL_DIR}/e0_dense_topk_results.json")
e1_results = load_results(f"{RETRIEVAL_DIR}/e1_bm25_topk_results.json")
e2_results = load_results(f"{RETRIEVAL_DIR}/e2_rrf_topk_results.json")
e5_results = load_results(f"{RETRIEVAL_DIR}/e5_vlm_rerank_topk_results.json")

e0_metrics = load_metrics(f"{RETRIEVAL_DIR}/e0_dense_metrics.json")
e1_metrics = load_metrics(f"{RETRIEVAL_DIR}/e1_bm25_metrics.json")
e2_metrics = load_metrics(f"{RETRIEVAL_DIR}/e2_rrf_metrics.json")
e5_metrics = load_metrics(f"{RETRIEVAL_DIR}/e5_vlm_rerank_metrics.json")

# E5 chỉ chạy trên subset — lấy đúng subset_idx đó để so E0 fair
e5_query_indices = {r["query_index"] for r in e5_results}
e0_subset_for_e5 = [r for r in e0_results if r["query_index"] in e5_query_indices]

print(f"E0: {len(e0_results):,} queries  |  E1: {len(e1_results):,}  "
      f"|  E2: {len(e2_results):,}  |  E5 subset: {len(e5_results):,}")


# ============================================================
# CELL 4 — BẢNG TỔNG HỢP RETRIEVAL (in đẹp)
# ============================================================
def compute_metrics(results, k_values=[1, 5, 10]):
    metrics, n = {}, len(results)
    for k in k_values:
        metrics[f"Recall@{k}"] = sum(
            1 for r in results
            if r["ground_truth_image_filename"] in [x["image_id"] for x in r["topk"][:k]]
        ) / n
    rr = [next((1/rank for rank, x in enumerate(r["topk"][:10], 1)
                if x["image_id"] == r["ground_truth_image_filename"]), 0.0)
          for r in results]
    metrics["MRR@10"] = float(np.mean(rr))
    ndcg = [next((1/np.log2(rank+1) for rank, x in enumerate(r["topk"][:10], 1)
                  if x["image_id"] == r["ground_truth_image_filename"]), 0.0)
            for r in results]
    metrics["NDCG@10"] = float(np.mean(ndcg))
    return metrics

# Tính lại E0 subset cho fair comparison với E5
e0_sub_m = compute_metrics(e0_subset_for_e5)

METRIC_NAMES = ["Recall@1", "Recall@5", "Recall@10", "MRR@10", "NDCG@10"]

print("\n" + "="*75)
print(f"{'BẢNG SO SÁNH RETRIEVAL':^75}")
print("="*75)
print(f"{'Metric':<12} {'E0 Dense':>10} {'E1 docTR':>10} {'E2 RRF':>10} "
      f"{'E5 VLM*':>10}  {'E0 sub*':>10}")
print("-"*75)
for m in METRIC_NAMES:
    print(f"{m:<12} {e0_metrics[m]:>10.4f} {e1_metrics[m]:>10.4f} "
          f"{e2_metrics[m]:>10.4f} {e5_metrics[m]:>10.4f}  {e0_sub_m[m]:>10.4f}")
print("="*75)
print("* E5 VLM và E0 sub: trên cùng subset (n=500 queries) — so sánh fair")
print("  E0/E1/E2: trên full 2000 queries")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 56.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00a 0:00:01
📦 Giải nén: /kaggle/input/notebooks/hnganhdng/e5-vlm-re-rank/docvqa_colqwen2_artifacts.zip
✅ corpus_id_to_idx.json
✅ e0_dense_topk_results.json
✅ e1_bm25_topk_results.json
✅ e2_rrf_topk_results.json
✅ e5_vlm_rerank_topk_results.json
✅ e0_dense_metrics.json
✅ e1_bm25_metrics.json
✅ e2_rrf_metrics.json
✅ e5_vlm_rerank_metrics.json

✅ Setup xong
E0: 2,000 queries  |  E1: 2,000  |  E2: 2,000  |  E5 subset: 500

                          BẢNG SO SÁNH RETRIEVAL                           
Metric         E0 Dense   E1 docTR     E2 RRF    E5 VLM*     E0 sub*
---------------------------------------------------------------------------
Recall@1         0.3775     0.1555     0.3385     0.2660      0.3460
Recall@5         0.5630     0.2375     0.51

In [ ]:
# ============================================================
# CELL 5 — ANLS FUNCTION + LOAD VLM
# ============================================================
def anls_score(prediction: str, ground_truth: str, threshold: float = 0.5) -> float:
    """
    Average Normalized Levenshtein Similarity — chuẩn chính thức DocVQA.
    NLS = 1 - edit_distance(pred, gt) / max(len(pred), len(gt))
    Nếu NLS < threshold → coi như 0 (model không trả lời được)
    """
    pred = prediction.strip().lower()
    gt   = ground_truth.strip().lower()
    if len(pred) == 0 and len(gt) == 0:
        return 1.0
    if len(pred) == 0 or len(gt) == 0:
        return 0.0
    max_len = max(len(pred), len(gt))
    nls = 1.0 - levenshtein_distance(pred, gt) / max_len
    return nls if nls >= threshold else 0.0
 
 
def mean_anls(predictions: list, ground_truths: list) -> float:
    scores = [anls_score(p, g) for p, g in zip(predictions, ground_truths)]
    return float(np.mean(scores))
 
 
# Load Qwen2-VL để generate answers
VLM_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
 
print(f"\nĐang load {VLM_MODEL_NAME} để generate answers...")
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_NAME, torch_dtype=dtype, device_map=None
).to(device).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
print(f"✅ Model loaded (device={device})")
 
print("Đang load dataset gốc để lấy ảnh...")
full_ds = load_dataset(DATASET_NAME, split="train")
print(f"✅ full_ds: {len(full_ds):,} records")
 
 
def safe_load_image(img, max_side=1024):
    try:
        img = img.convert("RGB")
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side), Image.LANCZOS)
        return img
    except Exception:
        return Image.new("RGB", (224, 224), color=(255, 255, 255))
 
 
def vlm_answer(image: Image.Image, query_text: str) -> str:
    """Dùng VLM đọc ảnh và trả lời câu hỏi — generate ngắn gọn."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":
             f"Look at this document image and answer the question concisely.\n"
             f"Question: {query_text}\nAnswer:"},
        ],
    }]
    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt], images=[image],
        return_tensors="pt", padding=True
    ).to(device)
    with torch.inference_mode():
        out = vlm_model.generate(**inputs, max_new_tokens=32, do_sample=False)
    # Chỉ lấy phần generated (bỏ phần prompt)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return vlm_processor.decode(generated, skip_special_tokens=True).strip()
 
 
# ============================================================
# CELL 6 — CHẠY END-TO-END QA TRÊN SUBSET (dùng cùng 500 queries E5)
# Với mỗi strategy, lấy top-1 image → VLM generate answer → tính ANLS
# ============================================================
E2E_PATH    = f"{RETRIEVAL_DIR}/e2e_qa_results.json"
E2E_METRICS = f"{RETRIEVAL_DIR}/e2e_qa_metrics.json"
 
# Map query_index → e1, e2 results để tra nhanh
e1_map = {r["query_index"]: r for r in e1_results}
e2_map = {r["query_index"]: r for r in e2_results}
e5_map = {r["query_index"]: r for r in e5_results}
 
qa_records = []
for e0_item in tqdm(e0_subset_for_e5, desc="End-to-End QA (4 strategies)"):
    qi  = e0_item["query_index"]
    q   = e0_item["query"]
    gt  = e0_item["answer"]
    gt_img = e0_item["ground_truth_image_filename"]
 
    # Top-1 image của mỗi strategy
    top1 = {
        "E0": e0_item["topk"][0]["image_id"],
        "E1": e1_map[qi]["topk"][0]["image_id"],
        "E2": e2_map[qi]["topk"][0]["image_id"],
        "E5": e5_map[qi]["topk"][0]["image_id"],
    }
 
    answers = {}
    for label, img_id in top1.items():
        row_idx = corpus_id_to_idx[img_id]
        img     = safe_load_image(full_ds[row_idx]["image"])
        answers[label] = vlm_answer(img, q)
 
    qa_records.append({
        "query_index": qi,
        "query": q,
        "ground_truth": gt,
        "gt_image": gt_img,
        "top1": top1,
        "answers": answers,
        "retrieve_correct": {
            label: (img_id == gt_img) for label, img_id in top1.items()
        },
        "anls": {
            label: anls_score(ans, gt) for label, ans in answers.items()
        },
    })
 
    if len(qa_records) % 25 == 0:
        with open(E2E_PATH, "w", encoding="utf-8") as f:
            json.dump(qa_records, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache(); gc.collect()
 
# Save kết quả
with open(E2E_PATH, "w", encoding="utf-8") as f:
    json.dump(qa_records, f, ensure_ascii=False, indent=2)
 
# Tính mean ANLS từng strategy
e2e_metrics = {
    label: mean_anls(
        [r["answers"][label] for r in qa_records],
        [r["ground_truth"]   for r in qa_records]
    )
    for label in ["E0", "E1", "E2", "E5"]
}
 
with open(E2E_METRICS, "w", encoding="utf-8") as f:
    json.dump({"num_queries": len(qa_records), "anls": e2e_metrics,
               "created_at": datetime.now().isoformat()}, f, indent=2)
 
print("\n========== ANLS END-TO-END (n=500) ==========")
for label, score in e2e_metrics.items():
    print(f"   {label}: {score:.4f}")
 
 
# ============================================================
# CELL 7 — ERROR ANALYSIS: MA TRẬN 2×2 CHO TỪNG STRATEGY
# ============================================================
def error_matrix(qa_records, strategy):
    """
    Phân loại mỗi query vào 1 trong 4 ô:
    - TP: retrieve đúng + answer đúng (ANLS ≥ 0.5)
    - FP_lucky: retrieve SAI + answer đúng (VLM hallucinate đúng)
    - FN_reader: retrieve đúng + answer sai (lỗi reader)
    - TN: retrieve sai + answer sai
    """
    TP = FP_lucky = FN_reader = TN = 0
    for r in qa_records:
        correct_retrieve = r["retrieve_correct"][strategy]
        correct_answer   = r["anls"][strategy] >= 0.5
        if correct_retrieve and correct_answer:      TP += 1
        elif not correct_retrieve and correct_answer: FP_lucky += 1
        elif correct_retrieve and not correct_answer: FN_reader += 1
        else:                                         TN += 1
    return {"TP": TP, "FP_lucky": FP_lucky, "FN_reader": FN_reader, "TN": TN}
 
print("\n========== ERROR ANALYSIS: MA TRẬN 2×2 ==========")
print(f"{'':20} {'E0':>8} {'E1':>8} {'E2':>8} {'E5':>8}")
print("-" * 52)
matrices = {s: error_matrix(qa_records, s) for s in ["E0", "E1", "E2", "E5"]}
for label, desc in [
    ("TP",        "✅ Retrieve✓ Answer✓"),
    ("FN_reader", "⚠️  Retrieve✓ Answer✗"),
    ("FP_lucky",  "🎲 Retrieve✗ Answer✓"),
    ("TN",        "❌ Retrieve✗ Answer✗"),
]:
    row = f"{desc:<20}"
    for s in ["E0", "E1", "E2", "E5"]:
        row += f"{matrices[s][label]:>8}"
    print(row)
print("-" * 52)
print(f"{'Total':<20}" + "".join(f"{len(qa_records):>8}" for _ in range(4)))
 
 
# ============================================================
# CELL 8 — VISUALIZE: 3 BIỂU ĐỒ
# ============================================================
STRATEGIES   = ["E0", "E1", "E2", "E5"]
COLORS       = {"E0": "#378ADD", "E1": "#EF9F27", "E2": "#1D9E75", "E5": "#D85A30"}
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Multimodal RAG — Retrieval & QA Analysis", fontsize=14, fontweight="bold")
 
# ── Plot 1: Recall@K bar chart ─────────────────────────────
ax = axes[0]
metrics_dict = {
    "E0": e0_sub_m,       # subset để fair với E5
    "E1": compute_metrics([e1_map[r["query_index"]] for r in qa_records]),
    "E2": compute_metrics([e2_map[r["query_index"]] for r in qa_records]),
    "E5": e5_metrics,
}
x      = np.arange(3)
width  = 0.2
k_labels = ["Recall@1", "Recall@5", "Recall@10"]
for i, s in enumerate(STRATEGIES):
    vals = [metrics_dict[s][k] for k in k_labels]
    ax.bar(x + i*width, vals, width, label=s, color=COLORS[s])
ax.set_title("Recall@K Comparison (n=500 subset)")
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(["@1", "@5", "@10"])
ax.set_ylabel("Recall")
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(axis="y", alpha=0.3)
 
# ── Plot 2: ANLS bar chart ─────────────────────────────────
ax = axes[1]
anls_vals = [e2e_metrics[s] for s in STRATEGIES]
bars = ax.bar(STRATEGIES, anls_vals, color=[COLORS[s] for s in STRATEGIES])
ax.set_title("ANLS Score (End-to-End QA, n=500)")
ax.set_ylabel("ANLS")
ax.set_ylim(0, max(anls_vals) * 1.3)
for bar, val in zip(bars, anls_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
 
# ── Plot 3: 2×2 Error Matrix stacked bar ──────────────────
ax = axes[2]
n  = len(qa_records)
tp = [matrices[s]["TP"]/n for s in STRATEGIES]
fn = [matrices[s]["FN_reader"]/n for s in STRATEGIES]
fp = [matrices[s]["FP_lucky"]/n for s in STRATEGIES]
tn = [matrices[s]["TN"]/n for s in STRATEGIES]
 
x = np.arange(len(STRATEGIES))
b1 = ax.bar(x, tp, color="#2ECC71", label="✅ Retrieve✓ Answer✓")
b2 = ax.bar(x, fn, bottom=tp, color="#F39C12", label="⚠️ Retrieve✓ Answer✗")
b3 = ax.bar(x, fp, bottom=[a+b for a,b in zip(tp,fn)],
            color="#9B59B6", label="🎲 Retrieve✗ Answer✓ (lucky)")
b4 = ax.bar(x, tn, bottom=[a+b+c for a,b,c in zip(tp,fn,fp)],
            color="#E74C3C", label="❌ Retrieve✗ Answer✗")
ax.set_title("Error Analysis — 2×2 Matrix")
ax.set_xticks(x); ax.set_xticklabels(STRATEGIES)
ax.set_ylabel("Proportion"); ax.set_ylim(0, 1.0)
ax.legend(loc="upper right", fontsize=7)
ax.grid(axis="y", alpha=0.3)
 
plt.tight_layout()
plt.savefig(f"{WORK_ARTIFACT_DIR}/analysis_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Đã lưu: analysis_charts.png")
 
 
# ============================================================
# CELL 9 — BẢNG CUỐI TỔNG HỢP ĐẦY ĐỦ (in ra dán vào báo cáo)
# ============================================================
print("\n" + "="*70)
print(f"{'BẢNG KẾT QUẢ TỔNG HỢP CUỐI (n=500 subset)':^70}")
print("="*70)
print(f"{'Metric':<14} {'E0 Dense':>10} {'E1 docTR':>10} {'E2 RRF':>10} {'E5 VLM':>10}")
print("-"*70)
for m in METRIC_NAMES:
    print(f"{m:<14} {metrics_dict['E0'][m]:>10.4f} "
          f"{metrics_dict['E1'][m]:>10.4f} "
          f"{metrics_dict['E2'][m]:>10.4f} "
          f"{metrics_dict['E5'][m]:>10.4f}")
print("-"*70)
print(f"{'ANLS (E2E)':<14} {e2e_metrics['E0']:>10.4f} "
      f"{e2e_metrics['E1']:>10.4f} "
      f"{e2e_metrics['E2']:>10.4f} "
      f"{e2e_metrics['E5']:>10.4f}")
print("="*70)
 
print("\n========== KEY FINDINGS ==========")
best_r1  = max(STRATEGIES, key=lambda s: metrics_dict[s]["Recall@1"])
best_anls = max(STRATEGIES, key=lambda s: e2e_metrics[s])
print(f"Best Recall@1  : {best_r1}  ({metrics_dict[best_r1]['Recall@1']:.4f})")
print(f"Best ANLS      : {best_anls}  ({e2e_metrics[best_anls]:.4f})")
print(f"E0 Recall@50   : 0.7695  (pool E5 re-rank coverage)")
print(f"OCR bottleneck : E1 Recall@1={metrics_dict['E1']['Recall@1']:.4f} "
      f"— thấp hơn E0 {metrics_dict['E0']['Recall@1'] - metrics_dict['E1']['Recall@1']:.4f} điểm")
 
 
# ============================================================
# CELL 10 — ZIP LẠI
# ============================================================
import shutil
zip_path = "/kaggle/working/docvqa_colqwen2_artifacts.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/kaggle/working/docvqa_colqwen2_artifacts", "zip",
                    "/kaggle/working", "docvqa_colqwen2_artifacts")
print(f"\n✅ Đã zip: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
print("🎉 PIPELINE HOÀN CHỈNH — sẵn sàng viết báo cáo!")
 


Đang load Qwen/Qwen2-VL-2B-Instruct để generate answers...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.
